## 1. UNLMTD Ledger V2
Creates `furlenco_analytics.materialized_tables.unlmtd_ledger_V2` — the consolidated UNLMTD subscription ledger combining migrated, non-migrated, and renewal transactions.

In [0]:
%sql
-- ============================================================
-- UNLMTD Ledger v2 — Migrated from Redshift to Databricks
-- Catalog: furlenco_silver
-- Changes: nested CTEs flattened, json_extract_path_text →
--   get_json_object / colon-path, LISTAGG → collect_list,
--   GETDATE → CURRENT_TIMESTAMP, dateadd → ADD_MONTHS,
--   "Snapshot" SUBSTRING parsing → get_json_object
-- ============================================================
CREATE OR REPLACE TABLE furlenco_analytics.materialized_tables.unlmtd_ledger_V2 as 


WITH

-- [FLATTENED] Inner base for UNLMTD_PLAN_TENURE_LEDGER_1M
unlmtd_tenure_1m_base AS (
    SELECT
        entity_id,
        -- [MIGRATED] SUBSTRING/POSITION JSON parsing → colon-path (snapshot is VARIANT)
        snapshot:tenureStartDate::string   AS tenure_start_date,
        snapshot:tenureEndDate::string     AS tenure_end_date,
        snapshot:paymentDetails.id::string AS payment_id,
        snapshot:tenureInMonths::string    AS tenure_in_months,
        rank() OVER (PARTITION BY entity_id ORDER BY created_at DESC) AS ranker
    FROM furlenco_silver.order_management_systems_evolve.state_transitions
    WHERE entity_type = 'PLAN'
      AND to_state = 'ACTIVATED'
      AND from_state = 'FULFILLMENT_IN_PROGRESS'
),

UNLMTD_PLAN_TENURE_LEDGER_1M AS (
    SELECT * FROM unlmtd_tenure_1m_base WHERE ranker = 1
),

-- [FLATTENED] Inner base for UNLMTD_PLAN_TENURE_LEDGER_renewal
unlmtd_tenure_ren_base AS (
    SELECT
        entity_id,
        snapshot:tenureStartDate::string AS tenure_start_date,
        snapshot:tenureEndDate::string   AS tenure_end_date
    FROM furlenco_silver.order_management_systems_evolve.state_transitions
    WHERE entity_type = 'RENEWAL'
      AND to_state = 'APPLIED'
      AND from_state = 'FUTURE'
),

UNLMTD_PLAN_TENURE_LEDGER_renewal AS (
    SELECT * FROM unlmtd_tenure_ren_base
),

tax_rates AS (
    SELECT
        id AS plan_id,
        -- [MIGRATED] pricing_details is VARIANT → colon-path syntax
        pricing_details:taxDetails.cgstPercentage::string AS cgst_percentage,
        pricing_details:taxDetails.sgstPercentage::string AS sgst_percentage,
        pricing_details:taxDetails.igstPercentage::string AS igst_percentage
    FROM furlenco_silver.order_management_systems_evolve.plans
),

-- [FLATTENED] ss_id_mapper (was nested inside ledger_1)
ledger_1_ss_id_mapper AS (
    SELECT
        p.id AS customer_plan_id,
        spc.subscription_plan_id,
        CASE
            WHEN get_json_object(p.catalog_plan_snapshot, '$.lencoUnlmtdPlanReferences.subscriptionSaleId') = ''
            THEN get_json_object(p.lenco_unlmtd_plan_references, '$.subscriptionSaleId')
            ELSE get_json_object(p.catalog_plan_snapshot, '$.lencoUnlmtdPlanReferences.subscriptionSaleId')
        END AS subscriptionsaleid
    FROM furlenco_silver.order_management_systems_evolve.plans p
    INNER JOIN furlenco_silver.lenco_unlimited.subscription_sales ss
        ON ss.id = CASE
            WHEN get_json_object(p.catalog_plan_snapshot, '$.lencoUnlmtdPlanReferences.subscriptionSaleId') = ''
            THEN get_json_object(p.lenco_unlmtd_plan_references, '$.subscriptionSaleId')
            ELSE get_json_object(p.catalog_plan_snapshot, '$.lencoUnlmtdPlanReferences.subscriptionSaleId')
        END
    INNER JOIN furlenco_silver.pandora_evolve.subscription_plan_cycles spc
        ON spc.unlmtd_subscription_sale_id = ss.id
    WHERE p.is_migrated_for_evolve = 'true'
),

-- [FLATTENED] abc (was nested inside ledger_1)
ledger_1_abc AS (
    SELECT
        spc.id                                      AS spc_id,
        sp.user_id,
        u.identifier,
        u.contact_no,
        u.first_name,
        u.email_id,
        spc.subscription_plan_id,
        spc.state                                   AS spc_state,
        sp.activated_at,
        sp.cancelled_at,
        spc.start_date,
        CASE
            WHEN spc.state != 'cancelled' THEN spc.end_date
            -- [MIGRATED] dateadd('month',N,date)-1 → ADD_MONTHS(date,N) - INTERVAL 1 DAY
            ELSE ADD_MONTHS(spc.start_date, cp.tenure_in_months) - INTERVAL 1 DAY
        END                                         AS end_date,
        -- [MIGRATED] purchase_date is DATE in Databricks; interval addition promotes to TIMESTAMP
        spc.purchase_date + INTERVAL 330 MINUTES    AS purchase_date,
        ss.id                                       AS ss_id,
        cp.tenure_in_months,
        pl.name                                     AS plan_name,
        cppc.sale_price,
        pay.payment_amount                          AS payableafterpaymentoffers,
        pay.gringotts_payment_id,
        row_number() OVER (
            PARTITION BY sp.user_id, spc.subscription_plan_id
            ORDER BY spc.purchase_date
        )                                           AS row_num
    FROM furlenco_silver.pandora_evolve.subscription_plan_cycles spc
    INNER JOIN furlenco_silver.pandora_evolve.subscription_plans sp
        ON spc.subscription_plan_id = sp.id
    INNER JOIN furlenco_silver.lenco_unlimited.subscription_sales ss
        ON spc.unlmtd_subscription_sale_id = ss.id
    INNER JOIN furlenco_silver.lenco_unlimited.payments pay
        ON ss.id = pay.entity_id
        AND pay.entity_type = 'SubscriptionSale'
    LEFT JOIN furlenco_silver.panem_evolve.users u
        ON u.id = sp.user_id
    INNER JOIN furlenco_silver.lenco_unlimited.city_plans cp
        ON cp.id = spc.unlmtd_city_plan_id
    INNER JOIN furlenco_silver.lenco_unlimited.plans pl
        ON pl.id = cp.plan_id
    INNER JOIN furlenco_silver.lenco_unlimited.city_plan_price_codes cppc
        ON cppc.id = spc.unlmtd_city_plan_price_code_id
        AND pay.status = 'COMPLETE'
),

-- [FLATTENED] VAS subquery for ledger_1
ledger_1_vas AS (
    SELECT
        vap.subscription_plan_cycle_id,
        sp.name       AS vas_pack,
        sp.sale_price AS vas_price
    FROM furlenco_silver.pandora_evolve.value_added_packs vap
    INNER JOIN furlenco_silver.lenco_unlimited.service_packs sp
        ON sp.id = vap.unlmtd_reference_id
    WHERE sp.type != 'ZERO_PRICE'
    GROUP BY 1, 2, 3
),

-- [FLATTENED] base (was nested inside ledger_1)
ledger_1_base AS (
    SELECT
        a.*,
        c.medium_identifier                                       AS razorpay_payment_id,
        -- [MIGRATED] LISTAGG → array_join(collect_list(...))
        array_join(collect_list(b.vas_pack),  ', ')               AS vas_packs,
        array_join(collect_list(b.vas_price), ', ')               AS vas_prices
    FROM ledger_1_abc a
    LEFT JOIN ledger_1_vas b
        ON a.spc_id = b.subscription_plan_cycle_id
    LEFT JOIN furlenco_silver.cms.transactions c
        ON a.gringotts_payment_id = c.payment_id
    WHERE c.status IN (2, 3)
    GROUP BY a.spc_id, a.user_id, a.identifier, a.contact_no, a.first_name, a.email_id,
             a.subscription_plan_id, a.spc_state, a.activated_at, a.cancelled_at,
             a.start_date, a.end_date, a.purchase_date, a.ss_id, a.tenure_in_months,
             a.plan_name, a.sale_price, a.payableafterpaymentoffers, a.gringotts_payment_id,
             a.row_num, c.medium_identifier
),

ledger_1 AS (
    SELECT
        ssm.customer_plan_id,
        b.spc_id,
        b.user_id,
        b.identifier                                                            AS fur_id,
        b.contact_no,
        b.first_name,
        b.email_id,
        b.subscription_plan_id                                                  AS old_subscription_plan_id,
        b.spc_state,
        b.start_date                                                            AS tenure_start_date,
        b.end_date                                                              AS tenure_end_date,
        b.purchase_date,
        b.ss_id,
        b.activated_at,
        b.tenure_in_months,
        b.plan_name,
        CASE WHEN (b.sale_price >= 0 OR b.sale_price < 0)
             THEN CAST(b.sale_price AS STRING) END                              AS sale_price,
        CASE WHEN (b.payableafterpaymentoffers >= 0 OR b.payableafterpaymentoffers < 0)
             THEN CAST(b.payableafterpaymentoffers AS STRING) END               AS payableafterpaymentoffers,
        b.gringotts_payment_id,
        'migrated'                                                              AS mig_state,
        b.vas_packs,
        b.vas_prices
    FROM ledger_1_base b
    LEFT JOIN ledger_1_ss_id_mapper ssm
        ON ssm.subscription_plan_id = b.subscription_plan_id
    WHERE b.spc_id NOT IN ('12070','12131','12034','12119','7243','12127','12062','12060','12130')
      AND b.spc_state != 'future'
),

-- [FLATTENED] base for ledger_2
ledger_2_base AS (
    SELECT
        p.id                                                                        AS customer_plan_id,
        p.state                                                                     AS plan_status,
        p.created_at + INTERVAL 330 MINUTES                                         AS plan_created_date,
        u.contact_no,
        u.id                                                                        AS user_id,
        u.identifier                                                                AS fur_id,
        p.name                                                                      AS plan_name,
        u.first_name,
        u.last_name,
        u.email_id,
        p.created_at,
        CAST(ultl.tenure_in_months AS INT)                                          AS tenure_in_months,
        TO_DATE(REPLACE(ultl.tenure_start_date, '"', ''))                           AS tenure_start_date,
        TO_DATE(REPLACE(ultl.tenure_end_date,   '"', ''))                           AS tenure_end_date,
        sa.city,
        sa.address,
        -- [MIGRATED] lenco_unlmtd_plan_references is STRING → get_json_object
        get_json_object(p.lenco_unlmtd_plan_references, '$.cityPlanId')             AS city_plan_id,
        -- [MIGRATED] payment_details is VARIANT → colon-path
        COALESCE(pay.id, p.payment_details:id::int)                                 AS payment_id,
        p.pricing_details:basePrice::string                                         AS sale_price,
        p.payment_details:payableAfterPaymentOffers.total::string                   AS payableafterpaymentoffers,
        -- [MIGRATED] NULL guard: absent VARIANT key returns NULL not '', so COALESCE
        CASE WHEN COALESCE(p.payment_details:payable.total::string, '') = ''
             THEN 0 ELSE 1 END                                                      AS payableafterpaymentoffers_flag,
        pay.created_at + INTERVAL 330 MINUTES                                       AS purchase_date,
        p.activation_date                                                           AS activated_at,
        pay.user_action_type,
        'non-migrated'                                                              AS mig_state
    FROM furlenco_silver.order_management_systems_evolve.plans p
    LEFT JOIN furlenco_silver.panem_evolve.users u
        ON p.user_id = u.id
    LEFT JOIN furlenco_silver.order_management_systems_evolve.snapshotted_addresses sa
        ON p.snapshotted_delivery_address_id = sa.id
    LEFT JOIN UNLMTD_PLAN_TENURE_LEDGER_1M ultl
        ON ultl.entity_id = p.id
    LEFT JOIN furlenco_silver.gringotts_evolve.payments pay
        ON pay.id = ultl.payment_id
    LEFT JOIN furlenco_silver.gringotts_evolve.transactions grt
        ON pay.id = grt.payment_id
    WHERE p.is_migrated_for_evolve = 'false'
),

-- [FLATTENED] VAS for ledger_2
ledger_2_vas AS (
    SELECT
        entity_id,
        array_join(collect_list(vas_pack),  ', ') AS vas_packs,
        array_join(collect_list(vas_price), ', ') AS vas_prices
    FROM (
        SELECT
            entity_id,
            name                                                               AS vas_pack,
            -- [MIGRATED] pricing_details is VARIANT → colon-path syntax
            pricing_details:basePrice::string                                  AS vas_price
        FROM furlenco_silver.order_management_systems_evolve.value_added_services
        WHERE entity_type = 'PLAN'
          AND state IN ('ACTIVE', 'EXPIRED')
          AND user_action_type != 'RENEWAL_TRANSACTION'
    )
    GROUP BY 1
),

ledger_2 AS (
    SELECT
        f.customer_plan_id,
        NULL                      AS spc_id,
        f.user_id,
        f.fur_id,
        f.contact_no,
        f.first_name,
        f.email_id,
        NULL                      AS old_subscription_plan_id,
        f.plan_status             AS spc_state,
        f.tenure_start_date,
        f.tenure_end_date,
        f.purchase_date,
        NULL                      AS ss_id,
        f.activated_at,
        f.tenure_in_months,
        f.plan_name,
        f.sale_price,
        f.payableafterpaymentoffers,
        f.payment_id              AS gringotts_payment_id,
        f.mig_state,
        v.vas_packs,
        v.vas_prices
    FROM ledger_2_base f
    LEFT JOIN ledger_2_vas v ON f.customer_plan_id = v.entity_id
    WHERE f.payableafterpaymentoffers_flag = 1
),

-- [FLATTENED] VAS for ledger_renewals
ledger_renewals_vas AS (
    SELECT
        user_action_id,
        array_join(collect_list(vas_pack),  ', ') AS vas_packs,
        array_join(collect_list(vas_price), ', ') AS vas_prices
    FROM (
        SELECT
            user_action_id,
            name                                                               AS vas_pack,
            pricing_details:basePrice::string                                  AS vas_price
        FROM furlenco_silver.order_management_systems_evolve.value_added_services
        WHERE entity_type = 'PLAN'
          AND state IN ('ACTIVE', 'EXPIRED', 'PENDING')
          AND user_action_type = 'RENEWAL_TRANSACTION'
    )
    GROUP BY 1
),

-- [FLATTENED] Renewals subquery
ledger_renewals_data AS (
    SELECT
        r.entity_id                                                                  AS customer_plan_id,
        r.state                                                                      AS plan_status,
        r.renewal_transaction_id,
        p.created_at + INTERVAL 330 MINUTES                                          AS plan_created_date,
        u.contact_no,
        u.id                                                                         AS user_id,
        u.identifier                                                                 AS fur_id,
        p.name                                                                       AS plan_name,
        u.first_name,
        u.last_name,
        u.email_id,
        p.created_at,
        r.tenure_in_months,
        TO_DATE(REPLACE(uptlr.tenure_start_date, '"', ''))                           AS tenure_start_date,
        TO_DATE(REPLACE(uptlr.tenure_end_date,   '"', ''))                           AS tenure_end_date,
        sa.city,
        sa.address,
        get_json_object(p.lenco_unlmtd_plan_references, '$.cityPlanId')              AS city_plan_id,
        pay.id                                                                       AS payment_id,
        pay.created_at + INTERVAL 330 MINUTES                                        AS purchase_date,
        pay.user_action_type,
        -- [MIGRATED] pricing_details / payment_details are VARIANT in renewals
        r.pricing_details:basePrice::string                                          AS sale_price,
        r.payment_details:payableAfterPaymentOffers.total::string                    AS payableafterpaymentoffers,
        'non-migrated-ren'                                                           AS mig_state
    FROM furlenco_silver.order_management_systems_evolve.renewals r
    LEFT JOIN furlenco_silver.order_management_systems_evolve.plans p
        ON p.id = r.entity_id
    LEFT JOIN furlenco_silver.panem_evolve.users u
        ON p.user_id = u.id
    LEFT JOIN UNLMTD_PLAN_TENURE_LEDGER_renewal uptlr
        ON uptlr.entity_id = r.id
    LEFT JOIN furlenco_silver.order_management_systems_evolve.snapshotted_addresses sa
        ON p.snapshotted_delivery_address_id = sa.id
    -- [MIGRATED] payment_details is VARIANT → colon-path to extract payment id
    INNER JOIN furlenco_silver.gringotts_evolve.payments pay
        ON pay.id = r.payment_details:id::int AND pay.status = 2
    WHERE r.vertical = 'UNLMTD'
),

ledger_renewals AS (
    SELECT
        r.customer_plan_id,
        NULL                        AS spc_id,
        r.user_id,
        r.fur_id,
        r.contact_no,
        r.first_name,
        r.email_id,
        NULL                        AS old_subscription_plan_id,
        r.plan_status               AS spc_state,
        r.tenure_start_date,
        r.tenure_end_date,
        r.purchase_date,
        NULL                        AS ss_id,
        NULL                        AS activated_at,
        r.tenure_in_months,
        r.plan_name,
        r.sale_price,
        r.payableafterpaymentoffers,
        r.payment_id                AS gringotts_payment_id,
        r.mig_state,
        v.vas_packs,
        v.vas_prices
    FROM ledger_renewals_data r
    LEFT JOIN ledger_renewals_vas v
        ON r.renewal_transaction_id = v.user_action_id
),

final AS (
    SELECT * FROM ledger_2
    UNION
    SELECT * FROM ledger_1
    UNION
    SELECT * FROM ledger_renewals
),

final_2 AS (
    SELECT
        rank() OVER (PARTITION BY f.user_id ORDER BY f.purchase_date)              AS transactions,
        CASE
            WHEN rank() OVER (PARTITION BY f.user_id, f.customer_plan_id ORDER BY f.purchase_date) >= 2
            THEN 'RENEWAL' ELSE 'ACQ'
        END                                                                         AS ren_acq_flag,
        f.customer_plan_id,
        f.spc_id,
        f.user_id,
        f.fur_id,
        f.contact_no,
        f.first_name,
        f.email_id,
        f.old_subscription_plan_id,
        COALESCE(p.state, s.sp_state)                                               AS sp_state,
        f.spc_state,
        COALESCE(sp.cancelled_at, pc.created_at)                                    AS plan_cancelled_at,
        COALESCE(
            f.activated_at   + INTERVAL 330 MINUTES,
            b.activated_at   + INTERVAL 330 MINUTES
        )                                                                           AS activated_at,
        f.tenure_start_date + INTERVAL 330 MINUTES                                  AS tenure_start_date,
        f.tenure_end_date   + INTERVAL 330 MINUTES                                  AS tenure_end_date,
        f.purchase_date,
        f.ss_id,
        f.tenure_in_months,
        f.plan_name,
        f.sale_price,
        COALESCE(panm.amount, pam.amount)                                           AS final_amount,
        f.gringotts_payment_id,
        COALESCE(t.medium_identifier, cms_t.medium_identifier)                      AS medium_identifier,
        f.mig_state,
        f.vas_packs,
        f.vas_prices,
        COALESCE(panm.amount, pam.amount)                                           AS payment_amount,
        COALESCE(t.amount,    cms_t.amount)                                         AS transaction_amount,
        (COALESCE(panm.amount, pam.amount) - COALESCE(t.amount, cms_t.amount))      AS ncemi_discount
    FROM final f
    LEFT JOIN (
        SELECT customer_plan_id, MIN(tenure_start_date) AS activated_at
        FROM final GROUP BY 1
    ) b ON b.customer_plan_id = f.customer_plan_id
    LEFT JOIN (
        SELECT
            old_subscription_plan_id,
            CASE WHEN spc_state = 'APPLIED' THEN 'active' ELSE spc_state END AS sp_state
        FROM (
            SELECT
                old_subscription_plan_id,
                spc_state,
                rank() OVER (PARTITION BY old_subscription_plan_id ORDER BY purchase_date DESC) AS ranker
            FROM final
        ) WHERE ranker = 1
    ) s ON s.old_subscription_plan_id = f.old_subscription_plan_id
    LEFT JOIN (
        SELECT payment_id, medium_identifier, payment_gateway_amount AS amount
        FROM furlenco_silver.gringotts_evolve.transactions WHERE status = 2
    ) t ON t.payment_id = f.gringotts_payment_id AND f.mig_state != 'migrated'
    LEFT JOIN (
        -- [MIGRATED] cms.transactions.response_params is STRING → get_json_object
        SELECT payment_id, medium_identifier,
               get_json_object(response_params, '$.amount')::FLOAT * 0.01 AS amount
        FROM furlenco_silver.cms.transactions WHERE status = 2
    ) cms_t ON cms_t.payment_id = f.gringotts_payment_id AND f.mig_state = 'migrated'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.plan_cancellations pc
        ON pc.plan_id = f.customer_plan_id AND pc.state = 'COMPLETED'
    LEFT JOIN furlenco_silver.pandora_evolve.subscription_plans sp
        ON sp.id = f.old_subscription_plan_id
    LEFT JOIN furlenco_silver.gringotts_evolve.payments panm
        ON panm.id = f.gringotts_payment_id AND panm.status = 2 AND f.mig_state != 'migrated'
    LEFT JOIN furlenco_silver.cms.payments pam
        ON pam.id = f.gringotts_payment_id AND pam.status = 2 AND f.mig_state = 'migrated'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.plans p
        ON p.id = f.customer_plan_id
),

filter AS (
    SELECT DISTINCT * FROM final_2
)

SELECT
    f.*,
    tr.cgst_percentage,
    tr.igst_percentage,
    tr.sgst_percentage,
    -- [MIGRATED] GETDATE() → CURRENT_TIMESTAMP()
    CURRENT_TIMESTAMP() + INTERVAL 330 MINUTES AS ledger_updated_date
FROM filter f
LEFT JOIN tax_rates tr ON f.customer_plan_id = tr.plan_id

## 2. FB Revenue
Creates `furlenco_analytics.materialized_tables.fb_revenue` — Furbooks revenue table with tax breakdowns, discounts, and invoice-level details across all verticals.

In [0]:
%sql
-- ============================================================
-- fb_revenue — Migrated from Redshift to Databricks
-- Catalog: furlenco_silver
-- Changes:
--   • Stored procedure (plpgsql) stripped → standalone CTAS
--   • DROP+CREATE TABLE → CREATE OR REPLACE TABLE
--   • json_extract_path_text scalar fields
--       → pre-flattened VARIANT columns (monetary_components_taxableamount, _tax_total, etc.)
--   • json_extract_array_element_text on discounts array
--       → colon-path: monetary_components:discounts[N].field::STRING
--       (pre-flattened monetary_components_discounts is VARIANT, not ARRAY)
--   • is_igst_applicable comparison: cast VARIANT to DOUBLE before != 0
--   • Forward alias refs (ncemi_discount, code_N_amount) resolved via CTE
--   • nullif(...)::float → NULLIF(...)::DOUBLE
--   • Table casing: Invoiceable_Groups → invoiceable_groups, Cities → cities
-- ============================================================

-- CELL: Create or replace the fb_revenue table
-- Set <TARGET_CATALOG>.<TARGET_SCHEMA> before running.
-- Write-path rule: target must be under /Users/shyamu.yadav@furlenco.com/
-- Example: CREATE OR REPLACE TABLE furlenco_silver.shyamu_yadav.fb_revenue AS

CREATE OR REPLACE TABLE furlenco_analytics.materialized_tables.fb_revenue AS

-- [MIGRATED] CTE separates column definitions from the total_discounts computation
-- because Spark SQL does not allow a SELECT alias to be referenced in the same SELECT.
WITH base AS (
    SELECT
        ig.customer_identifier                                                           AS fur_id,
        rr.vertical,
        ig.user_id,
        c.name                                                                           AS city,
        rr.external_reference_type,
        rr.external_reference_id,
        COALESCE(o.created_at, r.created_at)                                             AS external_reference_created_at,
        rr.accountable_entity_id,
        rr.accountable_entity_type,
        COALESCE(i.name,  p.name,  a.name,  vas.name)                                   AS entity_name,
        COALESCE(i.state, p.state, a.state, vas.state)                                  AS current_state_product,
        -- [MIGRATED] json_extract_path_text(mc,'taxableAmount')
        --   → pre-flattened VARIANT column; avoids runtime JSON parsing
        rr.monetary_components_taxableamount::STRING                                     AS taxableAmount,
        COALESCE(i.charged_till_date, p.charged_till_date, a.charged_till_date, vas.end_date)
                                                                                         AS Charged_till_date,
        -- [MIGRATED] VARIANT value must be cast to DOUBLE before numeric comparison;
        --   COALESCE to 0 guards absent keys (NULL != 0 is NULL, not false)
        CASE WHEN COALESCE(rr.monetary_components_tax_breakup_igst_rate::DOUBLE, 0) != 0
             THEN 'yes' ELSE 'no' END                                                    AS is_igst_applicable,
        rr.monetary_components_tax_breakup_igst_rate::STRING                             AS igst_rate,
        rr.monetary_components_tax_breakup_sgst_rate::STRING                             AS sgst_rate,
        rr.monetary_components_tax_breakup_cgst_rate::STRING                             AS cgst_rate,
        rr.start_date,
        rr.end_date,
        rr.state                                                                         AS recognition_state,
        rr.recognised_at,
        rr.to_be_recognised_on,
        rr.recognition_type,
        rr.monetary_components_tax_total::STRING                                         AS total_tax,
        -- [MIGRATED] json_extract_array_element_text(json_extract_path_text(mc,'discounts'), N)
        --   → colon-path array indexing on VARIANT: monetary_components:discounts[N].field::STRING
        --   (pre-flattened monetary_components_discounts is VARIANT not ARRAY; [N] requires ARRAY type)
        --   Out-of-bounds index returns NULL → CASE evaluates safely
        CASE
            WHEN rr.monetary_components:discounts[0].code::STRING = 'NCEMI' THEN rr.monetary_components:discounts[0].amount::STRING
            WHEN rr.monetary_components:discounts[1].code::STRING = 'NCEMI' THEN rr.monetary_components:discounts[1].amount::STRING
            WHEN rr.monetary_components:discounts[2].code::STRING = 'NCEMI' THEN rr.monetary_components:discounts[2].amount::STRING
            WHEN rr.monetary_components:discounts[3].code::STRING = 'NCEMI' THEN rr.monetary_components:discounts[3].amount::STRING
            WHEN rr.monetary_components:discounts[4].code::STRING = 'NCEMI' THEN rr.monetary_components:discounts[4].amount::STRING
            WHEN rr.monetary_components:discounts[5].code::STRING = 'NCEMI' THEN rr.monetary_components:discounts[5].amount::STRING
            WHEN rr.monetary_components:discounts[6].code::STRING = 'NCEMI' THEN rr.monetary_components:discounts[6].amount::STRING
            ELSE '0'
        END                                                                              AS ncemi_discount,
        CASE WHEN rr.monetary_components:discounts[0].code::STRING != 'NCEMI'
             THEN rr.monetary_components:discounts[0].code::STRING   END                 AS code_1,
        CASE WHEN rr.monetary_components:discounts[0].code::STRING != 'NCEMI'
             THEN rr.monetary_components:discounts[0].amount::STRING END                 AS code_1_amount,
        CASE WHEN rr.monetary_components:discounts[1].code::STRING != 'NCEMI'
             THEN rr.monetary_components:discounts[1].code::STRING   END                 AS code_2,
        CASE WHEN rr.monetary_components:discounts[1].code::STRING != 'NCEMI'
             THEN rr.monetary_components:discounts[1].amount::STRING END                 AS code_2_amount,
        CASE WHEN rr.monetary_components:discounts[2].code::STRING != 'NCEMI'
             THEN rr.monetary_components:discounts[2].code::STRING   END                 AS code_3,
        CASE WHEN rr.monetary_components:discounts[2].code::STRING != 'NCEMI'
             THEN rr.monetary_components:discounts[2].amount::STRING END                 AS code_3_amount
    FROM furlenco_silver.furbooks_evolve.revenue_recognitions rr
    LEFT JOIN furlenco_silver.furbooks_evolve.invoiceable_groups ig
        ON rr.invoiceable_group_id = ig.id
    LEFT JOIN furlenco_silver.panem_evolve.cities c
        ON ig.city_id = c.id
    LEFT JOIN furlenco_silver.order_management_systems_evolve.items i
        ON i.id = rr.accountable_entity_id AND rr.accountable_entity_type = 'ITEM'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.plans p
        ON p.id = rr.accountable_entity_id AND rr.accountable_entity_type = 'PLAN'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.attachments a
        ON a.id = rr.accountable_entity_id AND rr.accountable_entity_type = 'ATTACHMENT'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.value_added_services vas
        ON vas.id = rr.accountable_entity_id AND rr.accountable_entity_type = 'VALUE_ADDED_SERVICES'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.orders o
        ON o.id = rr.external_reference_id AND rr.external_reference_type = 'ORDER'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.renewals r
        ON r.id = rr.external_reference_id AND rr.external_reference_type = 'RENEWAL'
    WHERE rr.deleted_at IS NULL
)

SELECT
    b.fur_id,
    b.vertical,
    b.user_id,
    b.city,
    b.external_reference_type,
    b.external_reference_id,
    b.external_reference_created_at,
    b.accountable_entity_id,
    b.accountable_entity_type,
    b.entity_name,
    b.current_state_product,
    b.taxableAmount,
    b.Charged_till_date,
    b.is_igst_applicable,
    b.igst_rate,
    b.sgst_rate,
    b.cgst_rate,
    b.start_date,
    b.end_date,
    b.recognition_state,
    b.recognised_at,
    b.to_be_recognised_on,
    b.recognition_type,
    b.total_tax,
    b.ncemi_discount,
    b.code_1,
    b.code_1_amount,
    b.code_2,
    b.code_2_amount,
    b.code_3,
    b.code_3_amount,
    -- [MIGRATED] nullif(alias,'')::float → NULLIF(b.col,'')::DOUBLE
    (COALESCE(NULLIF(b.ncemi_discount,  '')::DOUBLE, 0)
     + COALESCE(NULLIF(b.code_1_amount, '')::DOUBLE, 0)
     + COALESCE(NULLIF(b.code_2_amount, '')::DOUBLE, 0)
     + COALESCE(NULLIF(b.code_3_amount, '')::DOUBLE, 0)) AS total_discounts_incl_ncemi
FROM base b


## 3. FB Revenue Monthly Snapshot
Creates a timestamped snapshot table (e.g. `furlenco_finance.monthly_snapshots.fb_revenue_snapshotted_dynamic_month_with_year`) joining `fb_revenue` with `unlmtd_ledger_V2` for month-end reporting.

In [0]:
%sql
-- ============================================================
-- fb_revenue Monthly Snapshot — Migrated from Redshift to Databricks
-- Source procedure: fb_revenue_snapshot_monthly() (plpgsql)
-- Requires: DBR 13.3+
-- Changes:
--   • plpgsql DECLARE+EXECUTE → DECLARE OR REPLACE VARIABLE + IDENTIFIER()
--   • getdate() → from_utc_timestamp(current_timestamp(),'Asia/Kolkata')
--   • to_char(date,'Month') → DATE_FORMAT(...,'MMMM')  (no trim needed)
--   • to_char(date,'DD-Mon-YYYY') → DATE_FORMAT(date,'dd-MMM-yyyy')
--   • date_trunc('month',...)::date → CAST(DATE_TRUNC('MONTH',...) AS DATE)
--   • json_extract_path_text(payment_details,'id') → payment_details:id::INT (VARIANT)
--   • Nested subqueries → flattened CTEs
--   • Plan_Cancellations → plan_cancellations (lowercase)
-- ⚠ ACTION REQUIRED: Replace 'shyamu_yadav' (2 occurrences below) with the
--   schema where fb_revenue + unlmtd_ledger_v2 actually live.
-- ============================================================


-- ----------------------------------------------------------------------------
-- STEP 1: Compute dynamic table name in IST
-- Produces e.g.: furlenco_silver.shyamu_yadav.fb_revenue_snapshotted_october1_2024
-- MMMM = full month name without padding — no trim() needed unlike Redshift
-- ----------------------------------------------------------------------------

DECLARE OR REPLACE VARIABLE full_table_name STRING DEFAULT CONCAT(
  'furlenco_finance.monthly_snapshots.fb_revenue_snapshotted_',
  LOWER(DATE_FORMAT(CONVERT_TIMEZONE('UTC', 'Asia/Kolkata', CURRENT_TIMESTAMP()), 'MMMM')),  -- Full month name in lowercase
  DATE_FORMAT(CONVERT_TIMEZONE('UTC', 'Asia/Kolkata', CURRENT_TIMESTAMP()), 'd'), 
  '_',
  DATE_FORMAT(CONVERT_TIMEZONE('UTC', 'Asia/Kolkata', CURRENT_TIMESTAMP()), 'yyyy')   -- 4-digit year
);

-- ----------------------------------------------------------------------------
-- STEP 2: Create the snapshot using IDENTIFIER() for dynamic table naming
-- No EXECUTE IMMEDIATE, no string escaping — full query runs as normal SQL
-- ----------------------------------------------------------------------------

CREATE OR REPLACE TABLE IDENTIFIER(full_table_name) AS

WITH mit AS (
    -- [MIGRATED] json_extract_path_text(payment_details,'id') → payment_details:id::INT
    -- payment_details is VARIANT; cast to INT to match gringotts_evolve.transactions.payment_id (INT)
    SELECT a.id, a.trans_type, t.medium_identifier
    FROM (
        SELECT id, 'ORDER'   AS trans_type, TRY_CAST(try_variant_get(payment_details, '$.id', 'string') AS INT) AS payment_id
        FROM furlenco_silver.order_management_systems_evolve.orders
        UNION ALL
        SELECT id, 'RENEWAL' AS trans_type, TRY_CAST(try_variant_get(payment_details, '$.id', 'string') AS INT) AS payment_id
        FROM furlenco_silver.order_management_systems_evolve.renewals
    ) a
    LEFT JOIN (
        SELECT payment_id, medium_identifier
        FROM furlenco_silver.gringotts_evolve.transactions
        WHERE status = 2
    ) t ON t.payment_id = a.payment_id
),

npa AS (
    SELECT DISTINCT fur_id FROM furlenco_analytics.materialized_tables.npa_customers
),

inner_base AS (
    SELECT
        fb.*,
        COALESCE(uledger.medium_identifier, mit.medium_identifier)              AS medium_identifier,
        COALESCE(ritem.created_at, rat.created_at, pc.created_at)               AS Return_requested_date,
        COALESCE(ritem.fulfillment_date, rat.fulfillment_date, pct.updated_at)  AS Return_completed_date,
        CASE WHEN
            fb.recognised_at >= '2024-04-01'
            AND fb.recognised_at <= CAST(DATE_TRUNC('MONTH', from_utc_timestamp(current_timestamp(), 'Asia/Kolkata')) AS DATE)
            AND fb.end_date >= '2024-04-01'
            AND fb.recognised_at < fb.to_be_recognised_on
        THEN 'YES' ELSE 'NO' END                                                AS Future_recog_flag,
        CASE WHEN fb.vertical = 'FURLENCO_RENTAL' AND npa.fur_id IS NOT NULL
             THEN 'YES' ELSE 'NO' END                                           AS NPA_CUSTOMER
    FROM furlenco_analytics.materialized_tables.fb_revenue fb
    LEFT JOIN mit
        ON mit.id = fb.external_reference_id
        AND mit.trans_type = fb.external_reference_type
    LEFT JOIN furlenco_silver.order_management_systems_evolve.return_items ritem
        ON ritem.item_id = fb.accountable_entity_id
        AND fb.accountable_entity_type = 'ITEM'
        AND ritem.state != 'CANCELLED'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.plan_cancellations pc
        ON pc.plan_id = fb.accountable_entity_id
        AND fb.accountable_entity_type = 'PLAN'
        AND pc.state != 'CANCELLED'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.plan_cancellations pct
        ON pct.plan_id = fb.accountable_entity_id
        AND fb.accountable_entity_type = 'PLAN'
        AND pct.state = 'COMPLETED'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.return_attachments rat
        ON rat.attachment_id = fb.accountable_entity_id
        AND fb.accountable_entity_type = 'ATTACHMENT'
        AND rat.state != 'CANCELLED'
    LEFT JOIN (
        SELECT customer_plan_id, tenure_start_date, tenure_end_date, medium_identifier
        FROM furlenco_analytics.materialized_tables.unlmtd_ledger_v2
    ) uledger
        ON uledger.customer_plan_id = fb.accountable_entity_id
        AND fb.accountable_entity_type = 'PLAN'
        AND (fb.start_date) >= uledger.tenure_start_date
        AND (fb.end_date) >= uledger.tenure_start_date
        AND (fb.end_date) <= uledger.tenure_end_date
    LEFT JOIN npa ON npa.fur_id = fb.fur_id
    WHERE 1 = 1
    AND (
        (   fb.end_date   >= '2024-04-01'
        AND fb.start_date <= CAST(DATE_TRUNC('MONTH', from_utc_timestamp(current_timestamp(), 'Asia/Kolkata')) AS DATE)
        )
        OR
        (   fb.recognised_at >= '2024-04-01'
        AND fb.recognised_at <= CAST(DATE_TRUNC('MONTH', from_utc_timestamp(current_timestamp(), 'Asia/Kolkata')) AS DATE)
        AND fb.end_date      >= '2024-04-01'
        AND fb.recognised_at  < fb.to_be_recognised_on
        )
        OR
        (   fb.recognised_at >= '2024-04-01'
        AND fb.recognised_at <=  CAST(DATE_TRUNC('MONTH', from_utc_timestamp(current_timestamp(), 'Asia/Kolkata')) AS DATE)
        AND fb.recognised_at >= fb.to_be_recognised_on
        )
    )

)
SELECT DISTINCT
    a.*,
    vas.name                                                    AS vas_name,
    vas.state                                                   AS vas_current_state,
    COALESCE(i.hsn_code, atta.hsn_code)                        AS hsn_code,
    DATE_FORMAT(a.start_date, 'dd-MMM-yyyy')                   AS start_date_new,
    DATE_FORMAT(a.end_date,   'dd-MMM-yyyy')                   AS end_date_new
FROM inner_base a
LEFT JOIN furlenco_silver.order_management_systems_evolve.value_added_services vas
    ON a.accountable_entity_id = vas.id
    AND a.accountable_entity_type = 'VALUE_ADDED_SERVICE'
LEFT JOIN furlenco_silver.order_management_systems_evolve.items i
    ON a.accountable_entity_id = i.id
    AND a.accountable_entity_type = 'ITEM'
LEFT JOIN furlenco_silver.order_management_systems_evolve.attachments atta
    ON a.accountable_entity_id = atta.id
    AND a.accountable_entity_type = 'ATTACHMENT'
WHERE a.recognition_state NOT IN ('CANCELLED', 'INVALIDATED')
  AND a.NPA_CUSTOMER = 'NO'
--   AND a.vertical = 'UNLMTD'
-- AND accountable_entity_id = 2347168
;


-- ----------------------------------------------------------------------------
-- STEP 3: Confirm — visible in Databricks Workflow run log
-- ----------------------------------------------------------------------------

SELECT
    full_table_name AS created_table,
    current_timestamp() AS created_at;
